# 00 — Debug Historical Six-Month Forecasts

Inspect existing sales and rental forecasts and the historical data behind them. This notebook does not retrain models or change Delta tables or MLflow runs.

**Use from the fitted modeling notebook:** import this notebook into the same Databricks workspace folder as `08_XGB_Six_Month_Totals`. After that notebook's cell 4, run the following in a new cell:

```python
%run "./00_Debug_Historical"
```

That route copies the current fitted results into diagnostic tables. It also works after cells 6–7.

**Use standalone:** run all cells here after logging a fitted model in cell 6 of the modeling notebook. By default, this notebook finds the latest completed six-month-total parent run in your MLflow experiment. Set `DEBUG_MLFLOW_PARENT_RUN_ID` in cell 1 to inspect a specific saved run. A full `tables/predictions.csv.gz` artifact is another supported input; the 36-row confusion-count CSV is insufficient for game-level diagnostics.

The checks cover calendar gaps and six-month coverage, saved versus current targets, historical label availability, train/test errors, prediction spread and ranking, bracket discrimination, feature quality and shifts, and tuning outcomes. Sales are cumulative units at month 6; rent is the sum of monthly units across months 1–6 (unit-months).

**Share the results:** download `debug_summary_pdf` as CSV from the last cell. The table selector also exposes per-game errors, source gaps, feature details and tuning checks. Current source tables may have changed since a saved run; the report keeps this distinction explicit.


## 1. Settings

Set a parent MLflow run ID only if you want to pin a historical run. The defaults also work when invoked from the current fitted notebook session.


In [0]:
# CELL 1 — Diagnostic settings and helpers
# Runs read-only: source tables and existing fitted-run artifacts are only read.
import json
import tempfile
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DEBUG_HORIZON = 6
DEBUG_USE_CURRENT_SESSION = True
DEBUG_MLFLOW_PARENT_RUN_ID = None  # Pin a run ID to inspect a specific historical run.
DEBUG_MLFLOW_EXPERIMENT_PATH = None  # Default: /Users/<current_user>/AI_feature_extraction_forecasts
DEBUG_PREDICTIONS_CSV_PATH = None  # Optional FULL per-game predictions CSV, not the 36-row confusion CSV.
DEBUG_MODELING_CSV_PATH = None     # Optional modeling_data snapshot CSV.
DEBUG_LOAD_CURRENT_SOURCES = True
DEBUG_DEMAND_TABLE = "development_042_silver_sandbox.demand_forecast.demand_forecast_results"
DEBUG_AI_ATTR_TABLE = "development_042_silver_sandbox.demand_forecast.game_attr_ai_silver"
DEBUG_MAX_SOURCE_ROWS = 200_000  # Deduplicated rows; oversized sources are skipped, never sampled for target QA.

# Export the final debug_report_tables in cell 7's selector using Databricks Download CSV.
# No fitted parameters, training labels, existing predictions, Delta tables, or MLflow runs are changed.
debug_findings = []


def debug_add_finding(area, severity, evidence, action):
    item = {"area": str(area), "severity": str(severity).lower(),
            "evidence": str(evidence), "next_check": str(action)}
    if item not in debug_findings:
        debug_findings.append(item)


def debug_bool_series(values):
    """Accept booleans and their CSV spellings; never treat the string 'False' as True."""
    def convert(value):
        if pd.isna(value):
            return pd.NA
        if isinstance(value, (bool, np.bool_)):
            return bool(value)
        if isinstance(value, (int, float, np.integer, np.floating)) and value in (0, 1):
            return bool(value)
        if isinstance(value, str):
            text = value.strip().lower()
            if text in {"true", "1"}:
                return True
            if text in {"false", "0"}:
                return False
        return pd.NA
    return pd.Series(values, copy=False).map(convert).astype("boolean")


def debug_normalize_names(values):
    values = pd.Series(values, copy=False).astype("string").str.strip().str.lower().str.replace(r"\s+", " ", regex=True)
    return values.mask(values.isin(["", "none", "null", "nan", "na", "n/a"]))


def debug_target_type(values):
    return debug_normalize_names(values).map({
        "sales": "sales", "sale": "sales", "lease": "rental", "rent": "rental", "rental": "rental",
    })


if "display" not in globals():
    def display(value):
        print(value.to_string(index=False) if isinstance(value, pd.DataFrame) else value)

print("Historical debug: one six-month total per game and revenue type.")
print("The notebook reads fitted predictions and current sources; it does not train or save models.")


## 2. Fitted-run provenance

Load full per-game predictions, feature inputs and tuning artifacts. Inspect the selected run and cutoff before interpreting the results.


In [0]:
# CELL 2 — Load current fitted results or an existing historical MLflow snapshot
# Independent Databricks notebooks do not share Python variables. In standalone mode,
# this cell reads the most recent completed matching parent run from MLflow.
debug_predictions_pdf = pd.DataFrame()
debug_modeling_pdf = pd.DataFrame()
debug_tuning_summary_pdf = pd.DataFrame()
debug_tuning_trials_pdf = pd.DataFrame()
debug_tuning_folds_pdf = pd.DataFrame()
debug_training_config = {}
debug_raw_demand_pdf = pd.DataFrame()
debug_current_ai_attrs_pdf = pd.DataFrame()
debug_input_source = "unavailable"
debug_run_id = None
debug_run_start = None
debug_run_end = None
DEBUG_TEST_CUTOFF = None
debug_snapshot_status = []


def debug_copy_frame(variable):
    value = globals().get(variable)
    return value.copy(deep=True) if isinstance(value, pd.DataFrame) else pd.DataFrame()


# Explicit files/run IDs take priority over variables left in an interactive session.
if DEBUG_PREDICTIONS_CSV_PATH:
    try:
        debug_predictions_pdf = pd.read_csv(DEBUG_PREDICTIONS_CSV_PATH)
        if DEBUG_MODELING_CSV_PATH:
            debug_modeling_pdf = pd.read_csv(DEBUG_MODELING_CSV_PATH)
        debug_input_source = "explicit CSV snapshot"
    except Exception as exc:
        debug_add_finding("inputs", "error", f"Could not load CSV snapshot: {type(exc).__name__}: {exc}",
                          "Provide the full tables/predictions.csv.gz artifact or a complete per-game prediction export.")
elif DEBUG_USE_CURRENT_SESSION and not DEBUG_MLFLOW_PARENT_RUN_ID and isinstance(globals().get("xgb_predictions_pdf"), pd.DataFrame):
    debug_predictions_pdf = debug_copy_frame("xgb_predictions_pdf")
    debug_modeling_pdf = debug_copy_frame("model_pdf")
    debug_tuning_summary_pdf = debug_copy_frame("tuning_summary_pdf")
    debug_tuning_trials_pdf = debug_copy_frame("tuning_trials_pdf")
    debug_tuning_folds_pdf = debug_copy_frame("tuning_folds_pdf")
    debug_training_config = {
        "training_config": dict(globals().get("BASELINE_TRAINING_CONFIG", {})),
        "numeric_features": list(globals().get("numeric_features", [])),
        "categorical_features": list(globals().get("categorical_features", [])),
        "selected_parameters": dict(globals().get("best_params_by_target", {})),
    }
    debug_input_source = "current fitted notebook session (copied)"
    DEBUG_TEST_CUTOFF = pd.to_datetime(globals().get("TEST_CUTOFF"), errors="coerce")
    debug_run_id = globals().get("MLFLOW_PARENT_RUN_ID")
else:
    try:
        from mlflow.tracking import MlflowClient
        debug_client = MlflowClient()
        def has_prediction_snapshot(candidate):
            # The forecasting parent can itself be nested under an unrelated run.
            # Identify the snapshot owner by its artifacts, not by absence of a parent tag.
            return any(a.path == "tables/predictions.csv.gz"
                       for a in debug_client.list_artifacts(candidate.info.run_id, "tables"))
        debug_run_id = DEBUG_MLFLOW_PARENT_RUN_ID
        if debug_run_id:
            chosen = debug_client.get_run(debug_run_id)
            # Stop at the nearest ancestor that actually owns the per-game snapshot.
            seen_runs = set()
            while not has_prediction_snapshot(chosen):
                if chosen.info.run_id in seen_runs:
                    raise ValueError("MLflow parent-run cycle detected.")
                seen_runs.add(chosen.info.run_id)
                parent_id = chosen.data.tags.get("mlflow.parentRunId")
                if not parent_id:
                    raise ValueError("No full per-game prediction snapshot exists in the selected run or its ancestors.")
                chosen = debug_client.get_run(parent_id)
            debug_run_id = chosen.info.run_id
        else:
            experiment_path = DEBUG_MLFLOW_EXPERIMENT_PATH
            if not experiment_path:
                if "spark" not in globals():
                    raise ValueError("Set DEBUG_MLFLOW_EXPERIMENT_PATH when Spark current_user() is unavailable.")
                current_user = spark.sql("SELECT current_user() AS user_name").first()["user_name"]
                experiment_path = f"/Users/{current_user}/AI_feature_extraction_forecasts"
            experiment = debug_client.get_experiment_by_name(experiment_path)
            if experiment is None:
                raise ValueError(f"No MLflow experiment found at {experiment_path!r}.")
            candidates = debug_client.search_runs(
                [experiment.experiment_id], filter_string="tags.prediction_kind = 'six_month_total'",
                order_by=["attributes.start_time DESC"], max_results=100,
            )
            chosen = None
            for candidate in candidates:
                if candidate.info.status == "FINISHED" and has_prediction_snapshot(candidate):
                    chosen = candidate
                    break
            if chosen is None:
                raise ValueError("No completed six_month_total run with a full per-game snapshot found among the latest 100 matching runs.")
            debug_run_id = chosen.info.run_id
        debug_run_start = pd.to_datetime(chosen.info.start_time, unit="ms", utc=True)
        debug_run_end = pd.to_datetime(chosen.info.end_time, unit="ms", utc=True) if chosen.info.end_time else None
        if chosen.info.status != "FINISHED":
            debug_add_finding("inputs", "review", f"Selected MLflow run has status {chosen.info.status}.",
                              "Use a completed parent run for a full artifact snapshot.")
        DEBUG_TEST_CUTOFF = pd.to_datetime(chosen.data.params.get("test_cutoff"), errors="coerce")
        with tempfile.TemporaryDirectory(prefix="historical_debug_") as directory:
            available = {a.path for a in debug_client.list_artifacts(debug_run_id)}
            available.update(a.path for a in debug_client.list_artifacts(debug_run_id, "tables"))
            def read_run_json(path):
                if path not in available:
                    return {}
                local = debug_client.download_artifacts(debug_run_id, path, directory)
                return json.loads(Path(local).read_text())
            def read_run_frame(name, required=False):
                path = f"tables/{name}.csv.gz"
                if path not in available:
                    debug_snapshot_status.append({"artifact": path, "status": "missing"})
                    if required:
                        debug_add_finding("inputs", "error", f"Missing {path} in run {debug_run_id}.",
                                          "Run the debug notebook from the fitted session or select a run with per-game predictions.")
                    return pd.DataFrame()
                try:
                    local = debug_client.download_artifacts(debug_run_id, path, directory)
                    frame = pd.read_csv(local)
                    debug_snapshot_status.append({"artifact": path, "status": "loaded", "rows": len(frame)})
                    return frame
                except Exception as exc:
                    debug_snapshot_status.append({"artifact": path, "status": type(exc).__name__})
                    debug_add_finding("inputs", "review", f"Could not read {path}: {type(exc).__name__}: {exc}",
                                      "Inspect the selected run's artifact and retry with a valid CSV snapshot.")
                    return pd.DataFrame()
            debug_training_config = read_run_json("configuration.json")
            if not debug_training_config.get("selected_parameters"):
                debug_training_config["selected_parameters"] = read_run_json("selected_parameters.json")
            debug_predictions_pdf = read_run_frame("predictions", required=True)
            debug_modeling_pdf = read_run_frame("modeling_data")
            debug_tuning_summary_pdf = read_run_frame("tuning_summary")
            debug_tuning_trials_pdf = read_run_frame("tuning_trials")
            debug_tuning_folds_pdf = read_run_frame("tuning_folds")
        debug_input_source = "saved MLflow parent-run artifacts"
    except Exception as exc:
        debug_add_finding("inputs", "review", f"Historical run loading unavailable: {type(exc).__name__}: {exc}",
                          "Run %run ./00_Debug_Historical after cell 4 in 08_XGB_Six_Month_Totals, or set a parent run ID / full prediction CSV path.")

# Normalize CSV types without modifying any original fitted-run objects.
for name in ["debug_predictions_pdf", "debug_modeling_pdf"]:
    frame = globals()[name]
    if frame.empty:
        continue
    for flag in ["used_for_fit", "used_for_test", "is_observed", "has_complete_horizon", "target_is_imputed"]:
        if flag in frame:
            frame[flag] = debug_bool_series(frame[flag])
    for column in ["game_release_date", "sales_start_date", "rental_start_date", "source_start_date",
                   "target_window_end_month", "target_available_date", "first_observed_date"]:
        if column in frame:
            frame[column] = pd.to_datetime(frame[column], errors="coerce")
    if "game_key" in frame:
        frame["game_key"] = debug_normalize_names(frame["game_key"])
    if "target_type" in frame:
        frame["target_type"] = debug_target_type(frame["target_type"])

if not debug_predictions_pdf.empty:
    required = {"game_key", "target_type", "target_value", "prediction"}
    missing = required - set(debug_predictions_pdf.columns)
    configured_horizon = debug_training_config.get("training_config", {}).get("FORECAST_MONTHS")
    if missing or (configured_horizon is not None and int(configured_horizon) != DEBUG_HORIZON):
        debug_add_finding("inputs", "error", f"Prediction snapshot is incompatible: missing={sorted(missing)}, horizon={configured_horizon}.",
                          "Use the one-total-per-game six-month predictions snapshot. Confusion counts alone are insufficient.")
        debug_predictions_pdf = pd.DataFrame()
    elif "month_index" in debug_predictions_pdf:
        debug_add_finding("inputs", "error", "Prediction snapshot contains month_index, indicating an older monthly model.",
                          "Select the six_month_total parent run rather than a monthly-curve run.")
        debug_predictions_pdf = pd.DataFrame()

if debug_modeling_pdf.empty and not debug_predictions_pdf.empty:
    debug_modeling_pdf = debug_predictions_pdf.copy(deep=True)
    debug_add_finding("inputs", "info", "No separate modeling_data snapshot; predictions contain the fitted feature inputs.",
                      "Use the prediction snapshot for feature diagnostics.")

# Infer the cutoff only when not explicitly retained. Training ended labels alone
# do not establish an outer test cutoff, so we use the first test launch date.
if DEBUG_TEST_CUTOFF is None or pd.isna(DEBUG_TEST_CUTOFF):
    fitted_cutoff = debug_training_config.get("training_config", {}).get("TEST_CUTOFF")
    DEBUG_TEST_CUTOFF = pd.to_datetime(fitted_cutoff, errors="coerce")
if (DEBUG_TEST_CUTOFF is None or pd.isna(DEBUG_TEST_CUTOFF)) and not debug_predictions_pdf.empty:
    if {"split", "first_observed_date"}.issubset(debug_predictions_pdf.columns):
        DEBUG_TEST_CUTOFF = debug_predictions_pdf.loc[
            debug_predictions_pdf["split"].eq("test"), "first_observed_date"].min()
        if pd.notna(DEBUG_TEST_CUTOFF):
            debug_add_finding("split", "info", f"Historical cutoff inferred from earliest test game: {DEBUG_TEST_CUTOFF.date()}.",
                              "Pin the parent run to use its recorded cutoff for exact provenance.")
if DEBUG_TEST_CUTOFF is not None and pd.notna(DEBUG_TEST_CUTOFF):
    DEBUG_TEST_CUTOFF = pd.Timestamp(DEBUG_TEST_CUTOFF).tz_localize(None)
else:
    DEBUG_TEST_CUTOFF = None

provenance = {
    "input_source": debug_input_source, "mlflow_parent_run_id": debug_run_id,
    "run_started_utc": debug_run_start, "run_ended_utc": debug_run_end,
    "historical_test_cutoff": DEBUG_TEST_CUTOFF,
    "prediction_rows": len(debug_predictions_pdf), "modeling_rows": len(debug_modeling_pdf),
    "horizon_months": DEBUG_HORIZON,
}
debug_run_provenance_pdf = pd.DataFrame([provenance])
display(debug_run_provenance_pdf)
if debug_snapshot_status:
    display(pd.DataFrame(debug_snapshot_status))
if debug_predictions_pdf.empty:
    print("No compatible per-game predictions loaded. Current-source target checks can still run.")
else:
    experiments = debug_predictions_pdf.get("experiment_id", pd.Series(dtype="string")).dropna().unique()
    if len(experiments) > 1:
        debug_add_finding("inputs", "error", f"Prediction snapshot contains {len(experiments)} experiment IDs.",
                          "Select one experiment/run before interpreting pooled model diagnostics.")
        debug_predictions_pdf = pd.DataFrame()


## 3. Current historical sources

Read actuals and attribute identity metadata. Repeated experiment actuals are deduplicated for the audit; oversized inputs are skipped rather than partially sampled.


In [0]:
# CELL 3 — Read current source actuals and extraction identity metadata
# Current source tables can have changed since the fitted run. A difference from
# the saved target is evidence to investigate, not proof of a target-building bug.
debug_source_load_rows = []


def debug_load_source(table_name, desired_columns, label, required_columns):
    if "spark" not in globals():
        debug_add_finding("sources", "review", f"Spark is unavailable; skipped {label}.",
                          "Run in Databricks to compare the current tables with saved targets.")
        return pd.DataFrame()
    try:
        source = spark.table(table_name)
        missing = set(required_columns) - set(source.columns)
        if missing:
            raise ValueError(f"Missing required source columns: {sorted(missing)}")
        selected = [c for c in desired_columns if c in source.columns]
        # Forecast experiments repeat the same actuals. Deduplicate only the
        # selected actual fields here; conflicting values are retained for QA.
        frame = source.select(*selected).dropDuplicates().limit(DEBUG_MAX_SOURCE_ROWS + 1).toPandas()
        if len(frame) > DEBUG_MAX_SOURCE_ROWS:
            debug_add_finding("sources", "review", f"{label} exceeds {DEBUG_MAX_SOURCE_ROWS:,} distinct diagnostic rows; skipped complete-source target checks.",
                              "Increase DEBUG_MAX_SOURCE_ROWS after checking memory; a truncated sample cannot establish complete horizons.")
            debug_source_load_rows.append({"source": label, "table": table_name, "status": "too_large_skipped",
                                           "rows": None, "read_at_utc": pd.Timestamp.now(tz="UTC")})
            return pd.DataFrame()
        debug_source_load_rows.append({"source": label, "table": table_name, "status": "loaded",
                                       "rows": len(frame), "read_at_utc": pd.Timestamp.now(tz="UTC")})
        return frame
    except Exception as exc:
        debug_source_load_rows.append({"source": label, "table": table_name, "status": "unavailable",
                                       "rows": None, "read_at_utc": pd.Timestamp.now(tz="UTC")})
        debug_add_finding("sources", "review", f"Could not read {label}: {type(exc).__name__}: {exc}",
                          "Check the configured table path, schema and Unity Catalog read access.")
        return pd.DataFrame()


if DEBUG_LOAD_CURRENT_SOURCES:
    debug_raw_demand_pdf = debug_load_source(
        DEBUG_DEMAND_TABLE,
        ["target_game", "revenue_type", "month_index", "curve_start_month", "forecast_month",
         "actual_cumulative_sales", "actual_monthly_lease", "actual_is_imputed",
         "actual_sales_month_zero_filled"],
        "demand_actuals",
        ["target_game", "revenue_type", "month_index", "curve_start_month",
         "actual_cumulative_sales", "actual_monthly_lease", "actual_is_imputed"],
    )
    debug_current_ai_attrs_pdf = debug_load_source(
        DEBUG_AI_ATTR_TABLE,
        ["join_game_name_key", "game_name", "run_id", "saved_at", "parse_status",
         "game_category", "game_classification", "game_release_date",
         "to_sales_name_match", "to_rental_name_match"],
        "attribute_identity",
        ["join_game_name_key"],
    )
else:
    debug_add_finding("sources", "info", "Current-source reads disabled by DEBUG_LOAD_CURRENT_SOURCES.",
                      "Enable source reads to rebuild targets and inspect zero-filled calendar gaps.")

debug_source_load_pdf = pd.DataFrame(debug_source_load_rows)
if not debug_source_load_pdf.empty:
    display(debug_source_load_pdf)
if not debug_raw_demand_pdf.empty:
    print("Raw actuals include months beyond month 6 so a later observed month can confirm interior zero gaps.")
    print("The imputed curve padding is retained for inspection but does not establish eligibility.")


## 4. Targets, gaps and historical split

Rebuild targets independently with pandas. An observed month 8 can establish zero activity in missing months 5–7; padded forecasts cannot establish coverage. Rebuilding current data is a consistency check against the saved historical labels.


In [0]:
# Rebuild historical labels from current source actuals and check saved split rules.
import numpy as np
import pandas as pd

_DEBUG_NULL_NAMES = {"", "none", "null", "nan", "na", "n/a"}
_DEBUG_TARGET_KEYS = ["game_key", "target_type"]


def debug_normalize_name(values):
    values = values.astype("string").str.strip().str.lower().str.replace(r"\s+", " ", regex=True)
    return values.mask(values.isin(_DEBUG_NULL_NAMES))


def debug_parse_bool(values):
    """Parse bools from Spark/CSV without treating the string 'False' as true."""
    def one(value):
        if pd.isna(value):
            return pd.NA
        if isinstance(value, (bool, np.bool_)):
            return bool(value)
        if isinstance(value, (int, float, np.integer, np.floating)) and value in (0, 1):
            return bool(value)
        if isinstance(value, str):
            return {"true": True, "false": False, "1": True, "0": False}.get(value.strip().lower(), pd.NA)
        return pd.NA
    return values.map(one).astype("boolean")


def debug_month_start(values):
    return pd.to_datetime(values, errors="coerce").dt.to_period("M").dt.to_timestamp()


def debug_rebuild_historical_targets(raw, horizon=6):
    """Rebuild calendar-horizon totals without using any forecast/neighbor values.

    Conflicting source curves are excluded, not averaged. This is an independent
    pandas audit of the production Spark target preparation; no model is fitted.
    """
    required = {"target_game", "revenue_type", "month_index", "curve_start_month",
                "actual_cumulative_sales", "actual_monthly_lease", "actual_is_imputed"}
    missing = sorted(required - set(raw.columns))
    if missing:
        return {"missing_columns": missing, "targets": pd.DataFrame(), "coverage": pd.DataFrame(),
                "conflicts": pd.DataFrame(), "gaps": pd.DataFrame(), "monthly": pd.DataFrame(),
                "input_issues": pd.DataFrame()}
    rows = raw.copy()
    rows["game_key"] = debug_normalize_name(rows["target_game"])
    rows["target_type"] = rows["revenue_type"].astype("string").str.strip().str.lower().map({
        "sale": "sales", "sales": "sales", "lease": "rental", "rental": "rental", "rent": "rental"})
    month = pd.to_numeric(rows["month_index"], errors="coerce")
    good_month = month.notna() & np.isfinite(month) & month.ge(1) & month.eq(month.round())
    rows["source_start_date"] = debug_month_start(rows["curve_start_month"])
    input_issues = []
    for name, mask in [
        ("missing_or_invalid_game_name", rows["game_key"].isna()),
        ("unsupported_revenue_type", rows["target_type"].isna()),
        ("invalid_calendar_month_index", ~good_month),
        ("invalid_source_start_date", rows["source_start_date"].isna()),
    ]:
        input_issues.append({"issue": name, "source_rows": int(mask.sum())})
    valid_keys = rows["game_key"].notna() & rows["target_type"].notna() & good_month
    rows = rows.loc[valid_keys].copy()
    rows["month_index"] = month.loc[valid_keys].astype(int)
    if rows.empty:
        return {"missing_columns": [], "targets": pd.DataFrame(), "coverage": pd.DataFrame(),
                "conflicts": pd.DataFrame(), "gaps": pd.DataFrame(), "monthly": pd.DataFrame(),
                "input_issues": pd.DataFrame(input_issues)}
    sales = pd.to_numeric(rows["actual_cumulative_sales"], errors="coerce")
    rental = pd.to_numeric(rows["actual_monthly_lease"], errors="coerce")
    rows["raw_actual"] = sales.where(rows["target_type"].eq("sales"), rental)
    rows["imputed_flag"] = debug_parse_bool(rows["actual_is_imputed"])
    finite = pd.Series(np.isfinite(rows["raw_actual"].to_numpy(dtype=float, na_value=np.nan)), index=rows.index)
    rows["trusted_actual"] = rows["raw_actual"].where(finite & rows["imputed_flag"].eq(False).fillna(False))
    rows["unknown_actual"] = finite & rows["imputed_flag"].isna()
    month_keys = _DEBUG_TARGET_KEYS + ["month_index"]
    conflict_rows = []
    for key, group in rows.groupby(month_keys, sort=False, dropna=False):
        distinct = group["trusted_actual"].dropna().unique()
        if len(distinct) > 1:
            conflict_rows.append({"game_key": key[0], "target_type": key[1], "month_index": key[2],
                                  "conflict_type": "different_observed_actuals", "distinct_values": str(sorted(distinct.tolist()))})
    for key, group in rows.groupby(_DEBUG_TARGET_KEYS, sort=False, dropna=False):
        starts = group["source_start_date"].dropna().unique()
        if len(starts) > 1:
            conflict_rows.append({"game_key": key[0], "target_type": key[1], "month_index": np.nan,
                                  "conflict_type": "different_source_start_dates", "distinct_values": str(sorted(map(str, starts)))})
    conflicts = pd.DataFrame(conflict_rows, columns=_DEBUG_TARGET_KEYS + ["month_index", "conflict_type", "distinct_values"])
    conflict_pairs = set(map(tuple, conflicts[_DEBUG_TARGET_KEYS].drop_duplicates().to_numpy()))
    # Calendar date consistency is an audit only: the source may use its forecast
    # date for other curve kinds. Preserve the source month_index in the rebuild.
    if "forecast_month" in rows:
        calendar_month = debug_month_start(rows["forecast_month"])
        expected_index = ((calendar_month.dt.year - rows["source_start_date"].dt.year) * 12
                          + calendar_month.dt.month - rows["source_start_date"].dt.month + 1)
        comparable = calendar_month.notna() & rows["source_start_date"].notna()
        input_issues.append({"issue": "forecast_date_disagrees_with_calendar_index",
                             "source_rows": int((comparable & expected_index.ne(rows["month_index"])).sum())})
    monthly = (rows.groupby(month_keys, sort=False, dropna=False)
               .agg(trusted_actual=("trusted_actual", "max"), source_start_date=("source_start_date", "max"),
                    unknown_actual=("unknown_actual", "max"), repeated_source_rows=("month_index", "size"))
               .reset_index())
    target_rows, gap_rows = [], []
    for key, group in monthly.groupby(_DEBUG_TARGET_KEYS, sort=False, dropna=False):
        group = group.sort_values("month_index")
        start_values = group["source_start_date"].dropna()
        start = start_values.iloc[0] if len(start_values) else pd.NaT
        observed = group.loc[group["trusted_actual"].notna()]
        coverage = int(observed["month_index"].max()) if len(observed) else 0
        proof = observed.loc[observed["month_index"].ge(horizon), "month_index"]
        confirmation = int(proof.min()) if len(proof) else None
        inside = group.loc[group["month_index"].le(horizon)]
        has_unknown = bool(inside["unknown_actual"].any())
        conflict = key in conflict_pairs
        complete = bool(coverage >= horizon and pd.notna(start) and not has_unknown and not conflict)
        n_observed = int(inside["trusted_actual"].notna().sum())
        if conflict:
            reason = "conflicting_source_curves"
        elif pd.isna(start):
            reason = "missing_source_start"
        elif has_unknown:
            reason = "unknown_imputation_flag_in_horizon"
        elif coverage < horizon:
            reason = "coverage_below_six_months"
        else:
            reason = "eligible"
        endpoint = np.nan
        if complete:
            # Cumulative sales require the last trusted endpoint, not a sum of
            # cumulative rows. Monthly rental values are summed after zero fill.
            inside_observed = inside.loc[inside["trusted_actual"].notna()]
            if key[1] == "sales":
                endpoint = float(inside_observed["trusted_actual"].iloc[-1]) if len(inside_observed) else 0.0
            else:
                endpoint = float(inside_observed["trusted_actual"].sum())
        available = start + pd.DateOffset(months=confirmation) if pd.notna(start) and confirmation is not None else pd.NaT
        end = start + pd.DateOffset(months=horizon - 1) if pd.notna(start) else pd.NaT
        record = {"game_key": key[0], "target_type": key[1], "source_start_date": start,
                  "target_window_end_month": end, "target_available_date": available,
                  "coverage_months": coverage, "coverage_confirmation_index": confirmation,
                  "observed_months_in_horizon": n_observed,
                  "zero_filled_months": horizon - n_observed if complete else np.nan,
                  "has_complete_horizon": complete, "eligibility_reason": reason,
                  "rebuilt_target_value": endpoint}
        target_rows.append(record)
        if complete and n_observed < horizon:
            observed_by_index = inside.set_index("month_index")["trusted_actual"].reindex(range(1, horizon + 1))
            if key[1] == "sales":
                cumulative = observed_by_index.ffill().fillna(0.0)
                filled_monthly = cumulative.diff().fillna(cumulative.iloc[0])
            else:
                filled_monthly = observed_by_index.fillna(0.0)
            gap_rows.append({**record,
                             "observed_month_indices": ", ".join(map(str, observed["month_index"].tolist())),
                             "zero_filled_month_indices": ", ".join(map(str, observed_by_index.index[observed_by_index.isna()].tolist())),
                             "monthly_units_1_to_6": ", ".join(f"{x:g}" for x in filled_monthly)})
    targets = pd.DataFrame(target_rows)
    coverage_rows = []
    for kind, group in targets.groupby("target_type", sort=False):
        eligible = group["has_complete_horizon"]
        coverage_rows.append({"target_type": kind, "historical_games": len(group),
                              "eligible_six_month_games": int(eligible.sum()),
                              "incomplete_or_excluded_games": int((~eligible).sum()),
                              "eligible_games_with_zero_filled_gaps": int((eligible & group["zero_filled_months"].gt(0)).sum()),
                              "eligible_zero_totals": int((eligible & group["rebuilt_target_value"].eq(0)).sum()),
                              "eligible_negative_totals": int((eligible & group["rebuilt_target_value"].lt(0)).sum()),
                              "sales_units_or_rental_unit_months": "sales_units" if kind == "sales" else "rental_unit_months"})
    return {"missing_columns": [], "targets": targets, "coverage": pd.DataFrame(coverage_rows),
            "conflicts": conflicts, "gaps": pd.DataFrame(gap_rows), "monthly": monthly,
            "input_issues": pd.DataFrame(input_issues)}


def debug_audit_saved_splits(predictions, cutoff):
    """Return count evidence; never infer used-for-fit from split alone."""
    output = []
    if predictions is None or predictions.empty:
        return pd.DataFrame()
    p = predictions.copy()
    if not set(_DEBUG_TARGET_KEYS).issubset(p.columns):
        return pd.DataFrame([{"target_type": "all", "check": "missing_game_or_target_keys", "failures": len(p), "checked_rows": len(p)}])
    p["game_key"] = debug_normalize_name(p["game_key"])
    output.append({"target_type": "all", "check": "duplicate_game_target_rows",
                   "failures": int(p.duplicated(_DEBUG_TARGET_KEYS, keep=False).sum()), "checked_rows": len(p)})
    if "split" in p:
        crossing = p.groupby("game_key")["split"].nunique().gt(1)
        output.append({"target_type": "all", "check": "sales_and_rent_splits_disagree",
                       "failures": int(crossing.sum()), "checked_rows": int(p["game_key"].nunique())})
    cutoff = pd.to_datetime(cutoff, errors="coerce") if cutoff is not None else pd.NaT
    for kind, g in p.groupby("target_type", sort=False):
        fit = debug_parse_bool(g["used_for_fit"]).fillna(False) if "used_for_fit" in g else pd.Series(False, index=g.index)
        test = debug_parse_bool(g["used_for_test"]).fillna(False) if "used_for_test" in g else pd.Series(False, index=g.index)
        available = pd.to_datetime(g["target_available_date"], errors="coerce") if "target_available_date" in g else pd.Series(pd.NaT, index=g.index)
        complete = debug_parse_bool(g["has_complete_horizon"]).fillna(False) if "has_complete_horizon" in g else pd.Series(False, index=g.index)
        coverage = pd.to_numeric(g["coverage_months"], errors="coerce") if "coverage_months" in g else pd.Series(np.nan, index=g.index)
        scored = fit | test
        checks = [("fit_and_test_both_true", fit & test, pd.Series(True, index=g.index)),
                  ("used_target_missing_complete_horizon", scored & ~complete, scored),
                  ("used_target_coverage_below_six_months", scored & (coverage.isna() | coverage.lt(DEBUG_HORIZON)), scored)]
        if "target_is_imputed" in g:
            actual_flag = debug_parse_bool(g["target_is_imputed"])
            checks.append(("used_target_imputed_or_unknown", scored & ~actual_flag.eq(False).fillna(False), scored))
        if "target_value" in g:
            finite = pd.Series(np.isfinite(pd.to_numeric(g["target_value"], errors="coerce").to_numpy(dtype=float, na_value=np.nan)), index=g.index)
            checks.append(("used_target_nonfinite", scored & ~finite, scored))
        if "split" in g:
            checks.extend([("fit_flag_outside_train_split", fit & ~g["split"].eq("train"), fit),
                           ("test_flag_outside_test_split", test & ~g["split"].eq("test"), test)])
        if pd.notna(cutoff):
            checks.append(("fit_label_unavailable_at_cutoff", fit & (available.isna() | available.gt(cutoff)), fit))
            if "first_observed_date" in g:
                first_date = pd.to_datetime(g["first_observed_date"], errors="coerce")
                checks.extend([
                    ("fit_game_date_not_before_cutoff", fit & (first_date.isna() | first_date.ge(cutoff)), fit),
                    ("test_game_date_before_cutoff", test & (first_date.isna() | first_date.lt(cutoff)), test),
                ])
        if "target_window_end_month" in g:
            end = pd.to_datetime(g["target_window_end_month"], errors="coerce")
            checks.append(("label_availability_not_after_horizon_month", scored & (available.isna() | end.isna() | available.le(end)), scored))
            if "source_start_date" in g:
                start = debug_month_start(g["source_start_date"])
                actual_months = (end.dt.year - start.dt.year) * 12 + end.dt.month - start.dt.month + 1
                checks.append(("target_window_not_six_calendar_months", scored & (actual_months.isna() | actual_months.ne(DEBUG_HORIZON)), scored))
        for name, failures, checked in checks:
            output.append({"target_type": kind, "check": name, "failures": int(failures.sum()), "checked_rows": int(checked.sum())})
        mapping = "to_sales_name_match" if kind == "sales" else "to_rental_name_match"
        if mapping in g:
            alias = debug_normalize_name(g[mapping])
            fit_aliases = set(alias.loc[fit].dropna())
            test_aliases = set(alias.loc[test].dropna())
            output.append({"target_type": kind, "check": "source_aliases_cross_fit_test",
                           "failures": len(fit_aliases & test_aliases), "checked_rows": len(fit_aliases | test_aliases)})
        if "game_release_date" in g and "source_start_date" in g:
            release = pd.to_datetime(g["game_release_date"], errors="coerce")
            source = pd.to_datetime(g["source_start_date"], errors="coerce")
            years = ((release - source).dt.days / 365.25).abs()
            output.append({"target_type": kind, "check": "release_vs_source_start_over_two_years_review",
                           "failures": int(years.gt(2).sum()), "checked_rows": int(years.notna().sum())})
    return pd.DataFrame(output)


# Always define these tables so later summary/export cells can run after a skip.
debug_historical_targets_pdf = pd.DataFrame()
debug_target_coverage_pdf = pd.DataFrame()
debug_source_conflicts_pdf = pd.DataFrame()
debug_target_mismatches_pdf = pd.DataFrame()
debug_gap_examples_pdf = pd.DataFrame()
debug_split_audit_pdf = pd.DataFrame()
debug_source_input_issues_pdf = pd.DataFrame()
debug_rebuilt_monthly_pdf = pd.DataFrame()

_raw = globals().get("debug_raw_demand_pdf")
_saved = globals().get("debug_predictions_pdf")
if isinstance(_raw, pd.DataFrame) and not _raw.empty:
    _rebuilt = debug_rebuild_historical_targets(_raw, DEBUG_HORIZON)
    if _rebuilt["missing_columns"]:
        debug_add_finding("Targets", "warning", "Raw demand is missing columns: " + ", ".join(_rebuilt["missing_columns"]),
                          "Load the observed demand table used by 08_XGB_Six_Month_Totals.")
    else:
        debug_historical_targets_pdf = _rebuilt["targets"]
        debug_target_coverage_pdf = _rebuilt["coverage"]
        debug_source_conflicts_pdf = _rebuilt["conflicts"]
        debug_gap_examples_pdf = _rebuilt["gaps"]
        debug_source_input_issues_pdf = _rebuilt["input_issues"]
        debug_rebuilt_monthly_pdf = _rebuilt["monthly"]
        if not debug_source_conflicts_pdf.empty:
            debug_add_finding("Targets", "error", f"{len(debug_source_conflicts_pdf)} conflicting monthly actual/start-date groups; affected game/type targets were excluded.",
                              "Select the intended source experiment or resolve mappings before training.")
        if not debug_target_coverage_pdf.empty:
            print("Current-source six-month target coverage (sales units; rental unit-months):")
            display(debug_target_coverage_pdf)
            reasons = debug_historical_targets_pdf.groupby(["target_type", "eligibility_reason"]).size().rename("games").reset_index()
            display(reasons)
            print("Examples where later actuals confirm zero-activity gaps within the first six months:")
            display(debug_gap_examples_pdf.head(10))
        if not debug_source_input_issues_pdf.empty:
            display(debug_source_input_issues_pdf)
            for _issue in debug_source_input_issues_pdf.loc[debug_source_input_issues_pdf["source_rows"].gt(0)].to_dict("records"):
                debug_add_finding("Targets", "warning", f"{_issue['source_rows']} source rows: {_issue['issue']}.",
                                  "Review these source fields; the target reconstruction excludes invalid names/indices and requires an unambiguous start.")
        if isinstance(_saved, pd.DataFrame) and not _saved.empty and not debug_historical_targets_pdf.empty:
            _need = set(_DEBUG_TARGET_KEYS + ["target_value"])
            if _need.issubset(_saved.columns):
                _snapshot = _saved.copy()
                _snapshot["game_key"] = debug_normalize_name(_snapshot["game_key"])
                _snapshot = _snapshot.loc[~_snapshot.duplicated(_DEBUG_TARGET_KEYS, keep=False)].copy()
                _comparison = _snapshot.merge(debug_historical_targets_pdf, on=_DEBUG_TARGET_KEYS,
                                              how="left", suffixes=("_saved", "_rebuilt"), validate="one_to_one", indicator=True)
                _saved_value = pd.to_numeric(_comparison["target_value"], errors="coerce")
                _new_value = pd.to_numeric(_comparison["rebuilt_target_value"], errors="coerce")
                _match = np.isclose(_saved_value, _new_value, rtol=1e-7, atol=1e-6, equal_nan=True)
                _comparison["target_value_matches"] = _match
                _comparison["target_difference_current_minus_saved"] = _new_value - _saved_value
                _comparison["current_source_pair_found"] = _comparison["_merge"].eq("both")
                _different = ~_comparison["target_value_matches"]
                if "has_complete_horizon_saved" in _comparison:
                    _current_complete = _comparison["has_complete_horizon_rebuilt"].fillna(False).astype(bool)
                    _saved_complete = debug_parse_bool(_comparison["has_complete_horizon_saved"]).fillna(False)
                    _different |= _current_complete.ne(_saved_complete)
                _show = [c for c in ["game_key", "target_type", "target_value", "rebuilt_target_value",
                                     "target_difference_current_minus_saved", "target_value_matches", "current_source_pair_found",
                                     "has_complete_horizon_saved", "has_complete_horizon_rebuilt", "eligibility_reason",
                                     "source_start_date_saved", "source_start_date_rebuilt", "target_available_date_saved",
                                     "target_available_date_rebuilt", "coverage_months_saved", "coverage_months_rebuilt"] if c in _comparison]
                debug_target_mismatches_pdf = _comparison.loc[_different, _show].copy()
                if _different.any():
                    debug_add_finding("Targets", "warning", f"{int(_different.sum())}/{len(_comparison)} saved game/type labels differ from current-source reconstruction or current eligibility.",
                                      "Review examples and source/run timestamps. Current data can differ from the training snapshot; this is not automatically a target-code defect.")
                    display(debug_target_mismatches_pdf.head(30))
                else:
                    debug_add_finding("Targets", "info", f"All {len(_comparison)} comparable saved targets agree with current-source reconstruction.",
                                      "Target agreement supports checking model fit, features, and temporal changes next.")
else:
    debug_add_finding("Targets", "info", "Current raw demand was not loaded; current-source target reconstruction was skipped.",
                      "Enable current-source loading to audit calendar coverage, zero gaps, and saved-versus-current targets.")

if isinstance(_saved, pd.DataFrame) and not _saved.empty:
    debug_split_audit_pdf = debug_audit_saved_splits(_saved, globals().get("DEBUG_TEST_CUTOFF"))
    if not debug_split_audit_pdf.empty:
        print("Saved-run split and label-availability audit:")
        display(debug_split_audit_pdf)
        for _check in debug_split_audit_pdf.loc[debug_split_audit_pdf["failures"].gt(0)].to_dict("records"):
            _review_only = _check["check"] == "release_vs_source_start_over_two_years_review"
            debug_add_finding("Splits", "warning" if _review_only else "error",
                              f"{_check['target_type']}: {_check['check']} = {_check['failures']} ({_check['checked_rows']} checked).",
                              "Review commercial start versus release metadata." if _review_only else
                              "Review the flagged source identity, target completeness, and historical fit/test eligibility before comparing models.")


## 5. Prediction spread, ranking and errors

Training errors describe the fitted training rows and are not validation estimates. Compare them with held-out test errors, prediction spread, rank correlation, baseline performance and bracket recall.


In [0]:
# Compare fitted and held-out six-month totals. This cell only reads supplied snapshots.
_DEBUG_BRACKETS = ["bottom_20", "middle_60", "top_20"]
debug_regression_metrics_pdf = pd.DataFrame()
debug_prediction_spread_pdf = pd.DataFrame()
debug_bracket_summary_pdf = pd.DataFrame()
debug_bracket_recall_pdf = pd.DataFrame()
debug_bracket_confusion_pdf = pd.DataFrame()
debug_worst_games_pdf = pd.DataFrame()
debug_prediction_diagnostics_pdf = pd.DataFrame()


def debug_prediction_bool(values):
    """Parse Spark/CSV flags without interpreting the string 'False' as truthy."""
    text = pd.Series(values, copy=False).astype("string").str.strip().str.lower()
    return text.map({"true": True, "1": True, "1.0": True,
                     "false": False, "0": False, "0.0": False}).astype("boolean")


def debug_total_bracket(values, lower, upper):
    values = pd.to_numeric(pd.Series(values, copy=False), errors="coerce").astype(float)
    if lower == upper:
        return np.select([values.lt(lower), values.gt(upper)],
                         ["bottom_20", "top_20"], default="middle_60")
    return np.select([values.le(lower), values.gt(upper)],
                     ["bottom_20", "top_20"], default="middle_60")


def debug_safe_correlation(actual, predicted, rank=False):
    actual, predicted = pd.Series(actual, dtype=float), pd.Series(predicted, dtype=float)
    if len(actual) < 2 or actual.nunique() < 2 or predicted.nunique() < 2:
        return np.nan
    if rank:
        actual, predicted = actual.rank(method="average"), predicted.rank(method="average")
    return float(actual.corr(predicted))


def debug_error_record(actual, predicted):
    actual, predicted = np.asarray(actual, dtype=float), np.asarray(predicted, dtype=float)
    finite = np.isfinite(actual) & np.isfinite(predicted)
    actual, predicted = actual[finite], predicted[finite]
    if not len(actual):
        return {"games": 0, "MAE": np.nan, "RMSE": np.nan, "Bias": np.nan,
                "WAPE": np.nan, "R2": np.nan, "Pearson": np.nan, "Spearman": np.nan}
    error = predicted - actual
    sst, denominator = np.square(actual - actual.mean()).sum(), np.abs(actual).sum()
    return {
        "games": int(len(actual)), "MAE": float(np.abs(error).mean()),
        "RMSE": float(np.sqrt(np.square(error).mean())), "Bias": float(error.mean()),
        "WAPE": float(np.abs(error).sum() / denominator) if denominator > 0 else np.nan,
        "R2": float(1 - np.square(error).sum() / sst) if len(actual) > 1 and sst > 0 else np.nan,
        "Pearson": debug_safe_correlation(actual, predicted),
        "Spearman": debug_safe_correlation(actual, predicted, rank=True),
    }


def debug_audit_predictions(predictions):
    """Return diagnostic tables and per-game rows without fitting or changing models."""
    empty = {name: pd.DataFrame() for name in ["metrics", "spread", "brackets", "recall", "confusion", "worst", "rows"]}
    if not isinstance(predictions, pd.DataFrame) or predictions.empty:
        debug_add_finding("Predictions", "INFO", "No per-game prediction snapshot is available.",
                          "Load the parent MLflow run or provide its predictions.csv.gz; confusion counts alone cannot show prediction spread.")
        return empty
    required = {"game_key", "target_type", "target_value", "prediction", "has_complete_horizon",
                "coverage_months", "target_is_imputed"}
    missing = required - set(predictions.columns)
    if missing:
        debug_add_finding("Predictions", "WARNING", f"Prediction snapshot lacks: {sorted(missing)}.",
                          "Use the full tables/predictions.csv.gz artifact from the six-month-total run.")
        return empty
    frame = predictions.copy().reset_index(drop=True)
    frame["game_key"] = frame["game_key"].astype("string").str.strip()
    frame["target_type"] = frame["target_type"].astype("string").str.strip().str.lower().replace(
        {"sale": "sales", "lease": "rental", "rent": "rental"})
    key_valid = frame["game_key"].notna() & ~frame["game_key"].str.lower().isin(["", "none", "nan", "null"])
    key_valid &= frame["target_type"].isin(["sales", "rental"])
    if (~key_valid).any():
        debug_add_finding("Predictions", "WARNING", f"{int((~key_valid).sum())} prediction rows have missing/unsupported game or revenue keys.",
                          "Review game_key and target_type before scoring these rows.")
    frame = frame.loc[key_valid].copy()
    duplicate = frame.duplicated(["game_key", "target_type"], keep=False)
    if duplicate.any():
        debug_add_finding("Predictions", "ERROR", f"{int(duplicate.sum())} rows repeat a game/revenue pair; all repeated pairs are excluded from this audit.",
                          "Select a single model run and resolve duplicate scalar predictions.")
        frame = frame.loc[~duplicate].copy()
    for name in ["target_value", "prediction", "coverage_months", "baseline_prediction"]:
        if name in frame:
            frame[name] = pd.to_numeric(frame[name], errors="coerce").replace([np.inf, -np.inf], np.nan)
    frame["eligible_actual"] = (
        debug_prediction_bool(frame["has_complete_horizon"]).eq(True).fillna(False)
        & frame["coverage_months"].ge(DEBUG_HORIZON)
        & debug_prediction_bool(frame["target_is_imputed"]).eq(False).fillna(False)
        & frame["target_value"].notna()
    )
    if "is_observed" in frame:
        frame["eligible_actual"] &= debug_prediction_bool(frame["is_observed"]).eq(True).fillna(False)
    for flag in ["used_for_fit", "used_for_test"]:
        if flag in frame:
            frame[flag] = debug_prediction_bool(frame[flag]).eq(True).fillna(False).astype(bool)
        else:
            frame[flag] = False
            debug_add_finding("Predictions", "INFO", f"Snapshot has no {flag} flag; its corresponding split is not scored.",
                              "Use the full prediction snapshot to compare the actual fit and test cohorts.")
    overlap = frame["used_for_fit"] & frame["used_for_test"]
    if overlap.any():
        debug_add_finding("Split", "ERROR", f"{int(overlap.sum())} rows are marked used for both fit and test; they are excluded.",
                          "Review split flags and regenerate the prediction snapshot.")
    flag_invalid = overlap.copy()
    if "split" in frame:
        split = frame["split"].astype("string").str.strip().str.lower()
        inconsistent = ((frame["used_for_fit"] & ~split.eq("train").fillna(False))
                        | (frame["used_for_test"] & ~split.eq("test").fillna(False)))
        if inconsistent.any():
            debug_add_finding("Split", "ERROR", f"{int(inconsistent.sum())} rows have split labels inconsistent with fit/test flags; they are excluded.",
                              "Inspect split, used_for_fit and used_for_test in the saved run.")
        flag_invalid |= inconsistent
    flagged = frame["used_for_fit"] | frame["used_for_test"]
    ineligible_used = flagged & ~frame["eligible_actual"]
    if ineligible_used.any():
        debug_add_finding("Eligibility", "ERROR", f"{int(ineligible_used.sum())} fit/test rows fail the complete six-month actual eligibility checks.",
                          "Inspect coverage_months, has_complete_horizon and imputation flags before interpreting model scores.")
    missing_prediction = flagged & frame["eligible_actual"] & frame["prediction"].isna()
    if missing_prediction.any():
        debug_add_finding("Predictions", "WARNING", f"{int(missing_prediction.sum())} otherwise eligible fit/test rows lack finite predictions.",
                          "Inspect model output and exported numeric fields.")
    scored = frame.loc[frame["eligible_actual"] & ~flag_invalid & frame["prediction"].notna() & flagged].copy()
    if scored.empty:
        debug_add_finding("Predictions", "WARNING", "No unique, eligible fit/test rows have finite actuals and predictions.",
                          "Review the source, eligibility and split findings above.")
        return empty
    scored["evaluation_split"] = np.where(scored["used_for_test"], "test", "train")
    scored["actual_total"] = scored["target_value"]
    scored["predicted_total"] = scored["prediction"]
    scored["error"] = scored["prediction"] - scored["target_value"]
    scored["absolute_error"] = scored["error"].abs()
    scored["actual_bracket"] = pd.Series(pd.NA, index=scored.index, dtype="string")
    scored["predicted_bracket"] = pd.Series(pd.NA, index=scored.index, dtype="string")
    scored["baseline_bracket"] = pd.Series(pd.NA, index=scored.index, dtype="string")
    scored["bracket_correct"] = pd.Series(pd.NA, index=scored.index, dtype="boolean")
    scored["baseline_bracket_correct"] = pd.Series(pd.NA, index=scored.index, dtype="boolean")
    metrics, spreads, summaries, recalls, confusions = [], [], [], [], []
    predictors = [("xgboost", "prediction")]
    if "baseline_prediction" in scored:
        predictors.append(("training_mean", "baseline_prediction"))
    else:
        debug_add_finding("Baseline", "INFO", "No baseline_prediction column is present.",
                          "Load the full run snapshot to compare errors with the fitted training mean.")
    for (kind, split_name), cohort in scored.groupby(["target_type", "evaluation_split"], sort=True):
        for predictor, column in predictors:
            subset = cohort.loc[cohort[column].notna()]
            if subset.empty:
                continue
            record = {"target_type": kind, "evaluation_split": split_name, "predictor": predictor,
                      **debug_error_record(subset["target_value"], subset[column])}
            metrics.append(record)
            actual, predicted = subset["target_value"], subset[column]
            actual_std = float(actual.std(ddof=1)) if len(subset) >= 2 else np.nan
            pred_std = float(predicted.std(ddof=1)) if len(subset) >= 2 else np.nan
            ratio = pred_std / actual_std if np.isfinite(actual_std) and actual_std > 0 else np.nan
            spreads.append({
                "target_type": kind, "evaluation_split": split_name, "predictor": predictor,
                "games": len(subset), "actual_mean": float(actual.mean()), "actual_median": float(actual.median()),
                "actual_p20": float(actual.quantile(.2)), "actual_p80": float(actual.quantile(.8)),
                "actual_min": float(actual.min()), "actual_max": float(actual.max()), "actual_std": actual_std,
                "prediction_mean": float(predicted.mean()), "prediction_median": float(predicted.median()),
                "prediction_p20": float(predicted.quantile(.2)), "prediction_p80": float(predicted.quantile(.8)),
                "prediction_min": float(predicted.min()), "prediction_max": float(predicted.max()), "prediction_std": pred_std,
                "std_ratio_predicted_to_actual": ratio, "unique_predictions": int(predicted.nunique()),
                "unique_predictions_rounded_2dp": int(predicted.round(2).nunique()),
            })
            if predictor == "xgboost" and len(subset) >= 5 and np.isfinite(ratio) and ratio < .35:
                debug_add_finding("Prediction spread", "WARNING",
                                  f"{kind} {split_name}: prediction std is {ratio:.2f} times actual std across {len(subset)} games.",
                                  "Inspect train/test errors and rank correlation to distinguish range compression from weak ranking; this ratio alone does not prove underfitting.")
    metric_frame = pd.DataFrame(metrics)
    for kind, test in scored.loc[scored["evaluation_split"].eq("test")].groupby("target_type", sort=True):
        lower, upper = float(test["target_value"].quantile(.2)), float(test["target_value"].quantile(.8))
        index = test.index
        actual_bracket = debug_total_bracket(test["target_value"], lower, upper)
        scored.loc[index, "actual_bracket"] = actual_bracket
        for predictor, column in predictors:
            valid = test[column].notna()
            subset = test.loc[valid]
            if subset.empty:
                continue
            true = np.asarray(actual_bracket)[valid.to_numpy()]
            assigned = debug_total_bracket(subset[column], lower, upper)
            bracket_column = "predicted_bracket" if predictor == "xgboost" else "baseline_bracket"
            correct_column = "bracket_correct" if predictor == "xgboost" else "baseline_bracket_correct"
            scored.loc[subset.index, bracket_column] = assigned
            scored.loc[subset.index, correct_column] = true == assigned
            available_recalls = []
            for label in _DEBUG_BRACKETS:
                actual_count, predicted_count = int((true == label).sum()), int((assigned == label).sum())
                true_positive = int(((true == label) & (assigned == label)).sum())
                recall = true_positive / actual_count if actual_count else np.nan
                precision = true_positive / predicted_count if predicted_count else np.nan
                if actual_count:
                    available_recalls.append(recall)
                recalls.append({"target_type": kind, "predictor": predictor, "bracket": label,
                                "actual_games": actual_count, "predicted_games": predicted_count,
                                "correct_games": true_positive, "recall": recall, "precision": precision})
                for predicted_label in _DEBUG_BRACKETS:
                    count = int(((true == label) & (assigned == predicted_label)).sum())
                    confusions.append({"target_type": kind, "predictor": predictor,
                                       "actual_bracket": label, "predicted_bracket": predicted_label,
                                       "games": count, "fraction_of_actual_bracket": count / actual_count if actual_count else np.nan})
            summaries.append({"target_type": kind, "predictor": predictor, "games": len(subset),
                              "bracket_accuracy": float((true == assigned).mean()),
                              "balanced_bracket_accuracy": float(np.mean(available_recalls)),
                              "actual_brackets_present": len(available_recalls),
                              "predicted_brackets_present": int(pd.Series(assigned).nunique()),
                              "lower_actual_cutoff": lower, "upper_actual_cutoff": upper,
                              "cutoff_basis": "test_actual_total_p20_p80"})
            if predictor == "xgboost" and pd.Series(assigned).eq("middle_60").all():
                debug_add_finding("Brackets", "WARNING", f"{kind}: all {len(subset)} test predictions fall in middle_60.",
                                  "Inspect numeric spread and ranking; identical bracket assignments do not imply identical numeric predictions.")
        if len(test) < 15:
            debug_add_finding("Brackets", "INFO", f"{kind}: bracket diagnostics contain only {len(test)} eligible test games.",
                              "Treat individual tail-class scores as uncertain and compare additional chronological cohorts.")
    for kind, target_metrics in metric_frame.groupby("target_type", sort=True):
        test = target_metrics.loc[target_metrics["evaluation_split"].eq("test")].set_index("predictor")
        if {"xgboost", "training_mean"}.issubset(test.index):
            model, baseline = test.loc["xgboost"], test.loc["training_mean"]
            if model["games"] == baseline["games"] and model["RMSE"] >= baseline["RMSE"]:
                debug_add_finding("Regression", "WARNING",
                                  f"{kind}: test RMSE {model['RMSE']:,.2f} is no better than training-mean RMSE {baseline['RMSE']:,.2f}.",
                                  "Compare ranking, feature quality and train errors before choosing new model settings.")
        model_metrics = target_metrics.loc[target_metrics["predictor"].eq("xgboost")].set_index("evaluation_split")
        if {"train", "test"}.issubset(model_metrics.index):
            train_rmse, test_rmse = model_metrics.loc["train", "RMSE"], model_metrics.loc["test", "RMSE"]
            if test_rmse > 2 * max(train_rmse, 1e-12):
                debug_add_finding("Generalization", "INFO",
                                  f"{kind}: test RMSE is {test_rmse:,.2f}, versus in-sample training RMSE {train_rmse:,.2f}.",
                                  "Inspect scale/date differences between cohorts and chronological validation scores; a training/test error gap alone does not prove overfitting.")
    worst = scored.loc[scored["evaluation_split"].eq("test")].sort_values(
        ["target_type", "absolute_error"], ascending=[True, False]).groupby("target_type", sort=False).head(15)
    return {"metrics": metric_frame, "spread": pd.DataFrame(spreads), "brackets": pd.DataFrame(summaries),
            "recall": pd.DataFrame(recalls), "confusion": pd.DataFrame(confusions), "worst": worst,
            "rows": scored.reset_index(drop=True)}


_debug_prediction_results = debug_audit_predictions(debug_predictions_pdf)
debug_regression_metrics_pdf = _debug_prediction_results["metrics"]
debug_prediction_spread_pdf = _debug_prediction_results["spread"]
debug_bracket_summary_pdf = _debug_prediction_results["brackets"]
debug_bracket_recall_pdf = _debug_prediction_results["recall"]
debug_bracket_confusion_pdf = _debug_prediction_results["confusion"]
debug_worst_games_pdf = _debug_prediction_results["worst"]
debug_prediction_diagnostics_pdf = _debug_prediction_results["rows"]

print("Each scored row is one six-month game total. Training rows are in-sample diagnostics, not validation scores.")
print("WAPE is a fraction; bias = prediction − actual. Undefined R², correlations and spread ratios remain null.")
print("Bracket cutoffs come from held-out actual totals for retrospective evaluation; ties remain together.")
for _debug_title, _debug_table in [
    ("Regression errors and ranking", debug_regression_metrics_pdf),
    ("Prediction spread", debug_prediction_spread_pdf),
    ("Test bracket accuracy", debug_bracket_summary_pdf),
    ("Test bracket precision/recall", debug_bracket_recall_pdf),
]:
    if not _debug_table.empty:
        print(_debug_title)
        display(_debug_table)
if not debug_worst_games_pdf.empty:
    print("Largest held-out errors (up to 15 per revenue type)")
    _debug_show_columns = [name for name in [
        "game_key", "target_type", "target_value", "prediction", "baseline_prediction", "error",
        "absolute_error", "actual_bracket", "predicted_bracket", "bracket_correct", "source_start_date",
        "target_window_end_month", "coverage_months", "zero_filled_months", "game_release_date",
    ] if name in debug_worst_games_pdf]
    display(debug_worst_games_pdf[_debug_show_columns])

for _debug_kind, _debug_kind_rows in debug_prediction_diagnostics_pdf.groupby(
    "target_type", sort=True) if not debug_prediction_diagnostics_pdf.empty else []:
    _debug_test = _debug_kind_rows.loc[_debug_kind_rows["evaluation_split"].eq("test")]
    if _debug_test.empty:
        continue
    _debug_fig, _debug_axes = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)
    _debug_scatter, _debug_hist, _debug_residual, _debug_rank = _debug_axes.ravel()
    _debug_limits = []
    for _debug_split, _debug_color, _debug_marker in [("train", "#888888", "."), ("test", "#006BB6", "o")]:
        _debug_sample = _debug_kind_rows.loc[_debug_kind_rows["evaluation_split"].eq(_debug_split)]
        if not _debug_sample.empty:
            _debug_scatter.scatter(_debug_sample["target_value"], _debug_sample["prediction"],
                                   label=f"{_debug_split} (n={len(_debug_sample)})", alpha=.6,
                                   color=_debug_color, marker=_debug_marker)
            _debug_limits.extend(_debug_sample["target_value"].tolist() + _debug_sample["prediction"].tolist())
    _debug_lo, _debug_hi = min(_debug_limits), max(_debug_limits)
    if _debug_lo == _debug_hi:
        _debug_lo, _debug_hi = _debug_lo - 1, _debug_hi + 1
    _debug_scatter.plot([_debug_lo, _debug_hi], [_debug_lo, _debug_hi], "--", color="#333333", linewidth=1)
    _debug_scatter.set(title="Actual versus predicted total", xlabel="Actual total", ylabel="Predicted total")
    _debug_scatter.legend()
    _debug_bins = np.histogram_bin_edges(np.r_[_debug_test["target_value"], _debug_test["prediction"]], bins="auto")
    _debug_hist.hist(_debug_test["target_value"], bins=_debug_bins, density=True, alpha=.5, label="Actual", color="#E57A21")
    _debug_hist.hist(_debug_test["prediction"], bins=_debug_bins, density=True, alpha=.5, label="Predicted", color="#006BB6")
    _debug_hist.set(title="Held-out total distributions", xlabel="Six-month total", ylabel="Density")
    _debug_hist.legend()
    _debug_residual.scatter(_debug_test["target_value"], _debug_test["error"], color="#006BB6", alpha=.7)
    _debug_residual.axhline(0, color="#333333", linestyle="--", linewidth=1)
    _debug_residual.set(title="Held-out residuals", xlabel="Actual total", ylabel="Prediction − actual")
    _debug_sorted = _debug_test.sort_values(["target_value", "game_key"]).reset_index(drop=True)
    _debug_rank.plot(np.arange(1, len(_debug_sorted) + 1), _debug_sorted["target_value"], label="Actual", color="#E57A21")
    _debug_rank.plot(np.arange(1, len(_debug_sorted) + 1), _debug_sorted["prediction"], "o", markersize=4,
                     label="Predicted", color="#006BB6")
    if "baseline_prediction" in _debug_sorted:
        _debug_rank.plot(np.arange(1, len(_debug_sorted) + 1), _debug_sorted["baseline_prediction"], "--",
                         label="Training mean", color="#888888")
    _debug_rank.set(title="Held-out games ordered by actual total", xlabel="Game position (lowest to highest actual)", ylabel="Six-month total")
    _debug_rank.legend()
    for _debug_axis in _debug_axes.ravel():
        _debug_axis.grid(alpha=.2)
    _debug_fig.suptitle(f"{_debug_kind}: one {DEBUG_HORIZON}-month total per eligible game", fontsize=14)
    plt.show()


## 6. Features and tuning

Profile features using fitted training games, then compare held-out feature distributions. Associations are descriptive. Confirm whether chronological tuning ran and which parameters were selected.


In [0]:
# Feature quality, historical train/test differences, and tuning provenance.
# All associations below use training games only and are descriptive.
import json


def _debug_feature_frame(value):
    return value.copy() if isinstance(value, pd.DataFrame) else pd.DataFrame()


def _debug_feature_bool(values):
    """Read CSV/Spark booleans without treating the string 'False' as true."""
    values = pd.Series(values, copy=False)
    return values.map(lambda value: (
        True if value is True or isinstance(value, np.bool_) and bool(value) else
        False if value is False or isinstance(value, np.bool_) and not bool(value) else
        True if isinstance(value, (int, float, np.integer, np.floating)) and value == 1 else
        False if isinstance(value, (int, float, np.integer, np.floating)) and value == 0 else
        True if isinstance(value, str) and value.strip().lower() in {"true", "1"} else
        False if isinstance(value, str) and value.strip().lower() in {"false", "0"} else pd.NA
    )).astype("boolean")


def _debug_feature_categorical(series):
    values = series.astype("string").str.strip()
    return values.mask(values.str.lower().isin(["", "none", "null", "nan", "__missing__"]))


def _debug_feature_numeric(series):
    return pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan)


# Define every result even if source artifacts are unavailable.
debug_feature_quality_pdf = pd.DataFrame()
debug_feature_shift_pdf = pd.DataFrame()
debug_feature_associations_pdf = pd.DataFrame()
debug_duplicate_feature_vectors_pdf = pd.DataFrame()
debug_tuning_audit_pdf = pd.DataFrame()

_feature_predictions = _debug_feature_frame(globals().get("debug_predictions_pdf"))
_feature_modeling = _debug_feature_frame(globals().get("debug_modeling_pdf"))
_feature_config = globals().get("debug_training_config", {})
_feature_config = _feature_config if isinstance(_feature_config, dict) else {}
_feature_run_config = _feature_config.get("training_config", {})
_feature_run_config = _feature_run_config if isinstance(_feature_run_config, dict) else {}
_feature_available = set(_feature_predictions.columns) | set(_feature_modeling.columns)
_feature_fallback_numeric = [
    "reels", "paylines_or_ways", "persistence_score", "perceived_persistence_score", "has_parent_game",
] + sorted(name for name in _feature_available if
           name.startswith("theme_") and name.endswith("_score")
           or name.startswith("mechanic_") and name.endswith("_present"))
_feature_numeric = _feature_config.get("numeric_features")
_feature_categorical = _feature_config.get("categorical_features")
if not isinstance(_feature_numeric, list):
    _feature_numeric = _feature_fallback_numeric
if not isinstance(_feature_categorical, list):
    _feature_categorical = ["game_category", "game_classification"]
_feature_numeric = list(dict.fromkeys(name for name in _feature_numeric if isinstance(name, str)))
_feature_categorical = list(dict.fromkeys(name for name in _feature_categorical if isinstance(name, str)))
_feature_allowlist = _feature_numeric + _feature_categorical

# Start with the actual model-use flags. Enrich a thin prediction export from its
# matching modeling snapshot only when that snapshot has unique game/type keys.
_feature_base = _feature_predictions if not _feature_predictions.empty else _feature_modeling
_feature_base = _feature_base.copy()
_feature_keys = ["game_key", "target_type"]
if not _feature_base.empty and all(name in _feature_base for name in _feature_keys):
    supplement = [name for name in _feature_allowlist if name not in _feature_base and name in _feature_modeling]
    if supplement and all(name in _feature_modeling for name in _feature_keys):
        if _feature_modeling.duplicated(_feature_keys, keep=False).any():
            debug_add_finding("features", "warning", "Modeling snapshot has duplicate game/type keys; feature enrichment skipped.",
                              "Resolve duplicate identities before comparing feature rows.")
        else:
            _feature_base = _feature_base.merge(_feature_modeling[_feature_keys + supplement],
                                                 on=_feature_keys, how="left", validate="many_to_one")
    duplicate_rows = _feature_base.duplicated(_feature_keys, keep=False)
    if duplicate_rows.any():
        debug_add_finding("features", "warning", f"Excluded {int(duplicate_rows.sum())} duplicated game/type feature rows.",
                          "Select one experiment/run and retain one row per game/revenue type.")
        _feature_base = _feature_base.loc[~duplicate_rows].copy()

_feature_usable = [name for name in _feature_allowlist if name in _feature_base]
_feature_missing_columns = [name for name in _feature_allowlist if name not in _feature_base]
if _feature_missing_columns and not _feature_base.empty and _feature_config.get("numeric_features") is not None:
    debug_add_finding("features", "info", f"{len(_feature_missing_columns)} configured feature columns are absent from the loaded prediction/modeling artifacts.",
                      "Load the full modeling_data snapshot from the same parent run; profiles cover available columns only.")
_feature_quality_rows, _feature_shift_rows, _feature_association_rows, _feature_vector_rows = [], [], [], []
if _feature_base.empty or "target_type" not in _feature_base or not _feature_usable:
    debug_add_finding("features", "info", "No model-input features are available for profiling.",
                      "Load the predictions or modeling_data snapshot from the same MLflow parent run.")
else:
    if "used_for_fit" in _feature_base:
        _feature_base["_debug_fit"] = _debug_feature_bool(_feature_base["used_for_fit"]).fillna(False)
    else:
        split = _feature_base.get("split", pd.Series("", index=_feature_base.index)).astype("string")
        _feature_base["_debug_fit"] = split.eq("train").fillna(False)
        if "is_observed" in _feature_base:
            _feature_base["_debug_fit"] &= _debug_feature_bool(_feature_base["is_observed"]).fillna(False)
        if "has_complete_horizon" in _feature_base:
            _feature_base["_debug_fit"] &= _debug_feature_bool(_feature_base["has_complete_horizon"]).fillna(False)
        if "target_is_imputed" in _feature_base:
            _feature_base["_debug_fit"] &= _debug_feature_bool(_feature_base["target_is_imputed"]).eq(False).fillna(False)
        if "coverage_months" in _feature_base:
            _feature_base["_debug_fit"] &= _debug_feature_numeric(_feature_base["coverage_months"]).ge(globals().get("DEBUG_HORIZON", 6))
        cutoff = globals().get("DEBUG_TEST_CUTOFF")
        if cutoff is not None and not pd.isna(cutoff) and "target_available_date" in _feature_base:
            available = pd.to_datetime(_feature_base["target_available_date"], errors="coerce", utc=True)
            _feature_base["_debug_fit"] &= available.le(pd.to_datetime(cutoff, utc=True)).fillna(False)
        debug_add_finding("features", "info", "Training feature mask reconstructed because used_for_fit is unavailable.",
                          "Prefer the saved predictions snapshot for exact fitted-game membership; review the available eligibility fields.")
    if "used_for_test" in _feature_base:
        _feature_base["_debug_test"] = _debug_feature_bool(_feature_base["used_for_test"]).fillna(False)
    else:
        _feature_base["_debug_test"] = _feature_base.get("split", pd.Series("", index=_feature_base.index)).astype("string").eq("test").fillna(False)
        if "is_observed" in _feature_base:
            _feature_base["_debug_test"] &= _debug_feature_bool(_feature_base["is_observed"]).fillna(False)
    for target_type, group in _feature_base.groupby("target_type", dropna=False):
        train = group.loc[group["_debug_fit"]].copy()
        test = group.loc[group["_debug_test"]].copy()
        if train.empty:
            debug_add_finding("features", "info", f"{target_type}: no fitted training rows available.",
                              "Load a full predictions snapshot that includes used_for_fit rows.")
            continue
        for name in _feature_usable:
            numeric = name in _feature_numeric
            fit_values = _debug_feature_numeric(train[name]) if numeric else _debug_feature_categorical(train[name])
            test_values = _debug_feature_numeric(test[name]) if numeric else _debug_feature_categorical(test[name])
            observed = fit_values.dropna()
            distinct = int(observed.nunique())
            _feature_quality_rows.append({
                "target_type": target_type, "feature": name, "kind": "numeric" if numeric else "categorical",
                "training_games": len(train), "observed_training_values": len(observed),
                "missing_fraction": float(fit_values.isna().mean()), "n_unique": distinct,
                "variance": float(observed.var(ddof=0)) if numeric and len(observed) else np.nan,
                "zero_fraction": float(observed.eq(0).mean()) if numeric and len(observed) else np.nan,
                "is_all_missing": len(observed) == 0, "is_constant": distinct <= 1,
            })
            row = {"target_type": target_type, "feature": name, "kind": "numeric" if numeric else "categorical",
                   "training_games": len(train), "test_games": len(test),
                   "train_missing_fraction": float(fit_values.isna().mean()),
                   "test_missing_fraction": float(test_values.isna().mean()) if len(test) else np.nan,
                   "missing_fraction_change": float(test_values.isna().mean() - fit_values.isna().mean()) if len(test) else np.nan,
                   "train_min": np.nan, "train_max": np.nan, "test_out_of_range_count": None,
                   "test_out_of_range_fraction": np.nan, "unseen_test_categories": None,
                   "unseen_test_category_count": None, "unseen_test_category_fraction": np.nan}
            if numeric and len(observed):
                low, high = float(observed.min()), float(observed.max())
                out_of_range = test_values.notna() & (test_values.lt(low) | test_values.gt(high))
                row.update(train_min=low, train_max=high, test_out_of_range_count=int(out_of_range.sum()),
                           test_out_of_range_fraction=float(out_of_range.sum() / test_values.notna().sum()) if test_values.notna().any() else np.nan)
            elif not numeric:
                unseen = test_values.notna() & ~test_values.isin(set(observed))
                row.update(unseen_test_category_count=int(unseen.sum()),
                           unseen_test_category_fraction=float(unseen.sum() / test_values.notna().sum()) if test_values.notna().any() else np.nan,
                           unseen_test_categories=", ".join(sorted(test_values.loc[unseen].dropna().unique().tolist())[:10]))
            _feature_shift_rows.append(row)
            if numeric and "target_value" in train:
                y = _debug_feature_numeric(train["target_value"])
                paired = fit_values.notna() & y.notna()
                # This does not use held-out game outcomes for feature selection.
                if int(paired.sum()) >= 10 and fit_values.loc[paired].nunique() > 1 and y.loc[paired].nunique() > 1:
                    rho = float(fit_values.loc[paired].rank(method="average").corr(y.loc[paired].rank(method="average")))
                    _feature_association_rows.append({"target_type": target_type, "feature": name,
                                                      "training_pairs": int(paired.sum()), "spearman": rho,
                                                      "abs_spearman": abs(rho)})
        eligible_vectors = group.loc[group["_debug_fit"] | group["_debug_test"]].copy()
        if "game_key" in eligible_vectors and not eligible_vectors.empty:
            fingerprints = pd.DataFrame(index=eligible_vectors.index)
            for name in _feature_usable:
                values = _debug_feature_numeric(eligible_vectors[name]) if name in _feature_numeric else _debug_feature_categorical(eligible_vectors[name])
                fingerprints[name] = values.astype("string").fillna("<missing>")
            eligible_vectors["_debug_vector_id"] = pd.util.hash_pandas_object(fingerprints, index=False).map(lambda value: f"{int(value):016x}")
            for vector_id, members in eligible_vectors.groupby("_debug_vector_id"):
                if len(members) > 1:
                    _feature_vector_rows.append({
                        "target_type": target_type, "vector_id": vector_id, "total_games": len(members),
                        "training_games": int(members["_debug_fit"].sum()), "test_games": int(members["_debug_test"].sum()),
                        "identical_train_test": bool(members["_debug_fit"].any() and members["_debug_test"].any()),
                        "example_game_keys": ", ".join(members["game_key"].astype(str).head(8)),
                    })
    debug_feature_quality_pdf = pd.DataFrame(_feature_quality_rows)
    debug_feature_shift_pdf = pd.DataFrame(_feature_shift_rows)
    debug_feature_associations_pdf = pd.DataFrame(_feature_association_rows)
    debug_duplicate_feature_vectors_pdf = pd.DataFrame(_feature_vector_rows)
    if not debug_feature_quality_pdf.empty:
        print("Feature quality uses fitted training games only. Missing and constant features cannot distinguish those games.")
        display(debug_feature_quality_pdf.sort_values(["target_type", "is_all_missing", "is_constant", "missing_fraction"],
                                                      ascending=[True, False, False, False]))
        for target_type, group in debug_feature_quality_pdf.groupby("target_type"):
            bad = group.loc[group["is_all_missing"] | group["is_constant"]]
            if len(bad):
                debug_add_finding("features", "info", f"{target_type}: {len(bad)}/{len(group)} available features are constant or entirely missing in fitted training games.",
                                  "Review extraction coverage and remove redundant predictors within training validation if useful.")
    if not debug_feature_shift_pdf.empty:
        shifted = debug_feature_shift_pdf.loc[
            debug_feature_shift_pdf["missing_fraction_change"].abs().ge(0.20)
            | debug_feature_shift_pdf["test_out_of_range_fraction"].fillna(0).ge(0.20)
            | debug_feature_shift_pdf["unseen_test_category_fraction"].fillna(0).gt(0)]
        print("Train/test feature differences: raw out-of-range counts are measured before the fitted caps and scaling.")
        display(shifted.sort_values(["target_type", "missing_fraction_change"], ascending=[True, False]) if len(shifted) else debug_feature_shift_pdf.head(15))
        if len(shifted):
            debug_add_finding("features", "warning", f"{len(shifted)} feature/type pairs have >=20-point missingness change, >=20% out-of-range values, or unseen test categories.",
                              "Inspect cohort and extraction differences; these flags identify differences, not a proven cause of error.")
    if not debug_feature_associations_pdf.empty:
        debug_feature_associations_pdf = debug_feature_associations_pdf.sort_values(["target_type", "abs_spearman"], ascending=[True, False]).reset_index(drop=True)
        print("Largest training-only rank associations (at least 10 paired games). These are descriptive, correlated and unadjusted for multiple comparisons.")
        display(debug_feature_associations_pdf.groupby("target_type", sort=False).head(15))
    if not debug_duplicate_feature_vectors_pdf.empty:
        print("Identical recorded feature vectors can describe different games; matching vectors alone do not establish leakage.")
        display(debug_duplicate_feature_vectors_pdf.sort_values("total_games", ascending=False).head(20))
        debug_add_finding("features", "info", f"{len(debug_duplicate_feature_vectors_pdf)} groups of games share identical available recorded attributes.",
                          "Review whether the feature set contains enough information to distinguish commercially different games.")

# Tuning outcomes and selected parameters come from the same saved run.
_feature_tuning_summary = _debug_feature_frame(globals().get("debug_tuning_summary_pdf"))
_feature_tuning_trials = _debug_feature_frame(globals().get("debug_tuning_trials_pdf"))
_feature_tuning_folds = _debug_feature_frame(globals().get("debug_tuning_folds_pdf"))
_feature_selected_params = _feature_config.get("selected_parameters", {})
_feature_selected_params = _feature_selected_params if isinstance(_feature_selected_params, dict) else {}
_tuning_targets = set(_feature_selected_params)
for frame in [_feature_tuning_summary, _feature_tuning_trials, _feature_tuning_folds]:
    if "target_type" in frame:
        _tuning_targets.update(frame["target_type"].dropna().astype(str))
_tuning_rows = []
for target_type in sorted(_tuning_targets):
    summary = (_feature_tuning_summary.loc[_feature_tuning_summary["target_type"].astype(str).eq(target_type)]
               if "target_type" in _feature_tuning_summary else pd.DataFrame())
    trials = (_feature_tuning_trials.loc[_feature_tuning_trials["target_type"].astype(str).eq(target_type)]
              if "target_type" in _feature_tuning_trials else pd.DataFrame())
    folds = (_feature_tuning_folds.loc[_feature_tuning_folds["target_type"].astype(str).eq(target_type)]
             if "target_type" in _feature_tuning_folds else pd.DataFrame())
    row = {"target_type": target_type, "status": "unavailable", "recorded_trial_rows": len(trials),
           "recorded_fold_rows": len(folds), "requested_trials": _feature_run_config.get("TUNING_TRIALS"),
           "selected_trial": None, "validation_improvement_pct": None}
    if not summary.empty:
        row.update(summary.iloc[0].to_dict())
    params = _feature_selected_params.get(target_type)
    if not isinstance(params, dict) and not trials.empty and "selected" in trials and "params_json" in trials:
        chosen = trials.loc[_debug_feature_bool(trials["selected"]).fillna(False)]
        if not chosen.empty:
            try:
                params = json.loads(chosen["params_json"].iloc[0])
            except (ValueError, TypeError):
                params = None
    params = params if isinstance(params, dict) else {}
    for name in ["n_estimators", "learning_rate", "max_depth", "min_child_weight", "reg_lambda", "reg_alpha", "gamma"]:
        row[name] = params.get(name)
    row["selected_parameters_json"] = json.dumps(params, sort_keys=True, default=str)
    _tuning_rows.append(row)
    if row["status"] in {"insufficient_validation_history", "disabled"}:
        debug_add_finding("tuning", "warning", f"{target_type}: tuning status is {row['status']}; configured parameters were retained.",
                          "Review the saved chronological fold counts and history; do not assume the requested trial budget ran.")
    child_weight = pd.to_numeric(pd.Series([params.get("min_child_weight")]), errors="coerce").iloc[0]
    l2_penalty = pd.to_numeric(pd.Series([params.get("reg_lambda")]), errors="coerce").iloc[0]
    if params and (child_weight >= 10 or l2_penalty >= 10):
        debug_add_finding("tuning", "info", f"{target_type}: selected min_child_weight={params.get('min_child_weight')}, reg_lambda={params.get('reg_lambda')}.",
                          "If training predictions are compressed and training error is high, compare lower regularization in chronological validation; these settings alone do not prove underfitting.")
debug_tuning_audit_pdf = pd.DataFrame(_tuning_rows)
if not debug_tuning_audit_pdf.empty:
    print("Recorded tuning status and final selected parameters:")
    display(debug_tuning_audit_pdf)
else:
    debug_add_finding("tuning", "info", "No tuning artifacts or selected-parameter metadata are available.",
                      "Load tuning_summary, tuning_trials, tuning_folds and configuration.json from the same parent run.")
if not _feature_tuning_trials.empty:
    order = [name for name in ["target_type", "selection_score", "trial"] if name in _feature_tuning_trials]
    display(_feature_tuning_trials.sort_values(order).groupby("target_type", sort=False).head(5)
            if order and "target_type" in _feature_tuning_trials else _feature_tuning_trials.head(10))
if not _feature_tuning_folds.empty:
    display(_feature_tuning_folds)

# Lightweight current-source identity QA is kept separate from the fitted snapshot.
_feature_current_attrs = _debug_feature_frame(globals().get("debug_current_ai_attrs_pdf"))
if not _feature_current_attrs.empty:
    key_column = next((name for name in ["game_key", "join_game_name_key"] if name in _feature_current_attrs), None)
    if key_column:
        normalized = _feature_current_attrs[key_column].astype("string").str.strip().str.lower().str.replace(r"\s+", " ", regex=True)
        invalid_keys = normalized.isna() | normalized.isin(["", "none", "null", "nan", "na", "n/a"])
        duplicates = normalized.loc[~invalid_keys].duplicated(keep=False)
        if invalid_keys.any() or duplicates.any():
            debug_add_finding("attribute_source", "info", f"Current attribute source: {int(invalid_keys.sum())} missing/invalid keys and {int(duplicates.sum())} rows on repeated normalized keys.",
                              "Check latest-valid-row selection and matching; raw extraction versions may legitimately repeat a key.")
    if "parse_status" in _feature_current_attrs:
        status_counts = _feature_current_attrs["parse_status"].fillna("<missing>").value_counts().rename_axis("parse_status").reset_index(name="rows")
        print("Current attribute-source extraction status (may differ from the saved training snapshot):")
        display(status_counts)


## 7. Findings and downloadable summary

Download `debug_summary_pdf` from the displayed table and share it for review. Use the table selector and rerun this last cell to inspect another diagnostic table.


In [0]:
# CELL 7 — Launch-cohort checks, findings, and one summary to download
# Cohort totals describe historical mix and do not prove why the model changed.
debug_cohort_summary_pdf = pd.DataFrame()
if not debug_predictions_pdf.empty and {"game_key", "target_type", "target_value", "first_observed_date"}.issubset(debug_predictions_pdf.columns):
    cohort = debug_predictions_pdf.copy(deep=True)
    dates = pd.to_datetime(cohort["first_observed_date"], errors="coerce")
    cohort["launch_year"] = dates.dt.year
    cohort["actual_total"] = pd.to_numeric(cohort["target_value"], errors="coerce")
    fit = debug_bool_series(cohort.get("used_for_fit", pd.Series(False, index=cohort.index))).fillna(False)
    test = debug_bool_series(cohort.get("used_for_test", pd.Series(False, index=cohort.index))).fillna(False)
    cohort["evaluation_role"] = np.select([fit.to_numpy(dtype=bool), test.to_numpy(dtype=bool)],
                                           ["fit", "test"], default="not_used")
    cohort = cohort.loc[~cohort.duplicated(["game_key", "target_type"], keep=False)]
    debug_cohort_summary_pdf = (cohort.groupby(["target_type", "evaluation_role", "launch_year"], dropna=False)
        .agg(games=("game_key", "nunique"), actual_mean=("actual_total", "mean"),
             actual_median=("actual_total", "median"), first_launch=("first_observed_date", "min"),
             last_launch=("first_observed_date", "max")).reset_index())
    display(debug_cohort_summary_pdf)
    if {"game_release_date", "source_start_date"}.issubset(cohort.columns):
        metadata_date = pd.to_datetime(cohort["game_release_date"], errors="coerce")
        source_date = pd.to_datetime(cohort["source_start_date"], errors="coerce")
        different = metadata_date.notna() & source_date.notna() & ((metadata_date - source_date).abs().dt.days > 365)
        if different.any():
            debug_add_finding("dates", "review", f"{int(different.sum())} game/type rows have metadata release and commercial source start more than one year apart.",
                              "Inspect whether source starts, game mapping, and release metadata refer to the intended commercial game/version.")
            display(cohort.loc[different, ["game_key", "target_type", "game_release_date", "source_start_date", "first_observed_date"]].head(20))

if not debug_findings:
    debug_add_finding("coverage", "info", "No automatic flags were generated for the available inputs.",
                      "Review the loaded-data provenance and plots; absent flags are not a validation certificate.")
debug_findings_pdf = pd.DataFrame(debug_findings)
severity_order = {"error": 0, "review": 1, "warning": 1, "info": 2, "pass": 3}
debug_findings_pdf = (debug_findings_pdf.assign(_priority=debug_findings_pdf["severity"].map(severity_order).fillna(2))
                      .sort_values(["_priority", "area"], kind="stable").drop(columns="_priority").reset_index(drop=True))

# A fixed long-format summary collects the main evidence into ONE exportable CSV.
# Per-game targets and feature-level details remain in the selector below.
summary_rows = []
summary_tables = {
    "regression": "debug_regression_metrics_pdf",
    "prediction_spread": "debug_prediction_spread_pdf",
    "bracket_summary": "debug_bracket_summary_pdf",
    "bracket_recall": "debug_bracket_recall_pdf",
    "target_coverage": "debug_target_coverage_pdf",
    "tuning": "debug_tuning_audit_pdf",
}
identity_columns = {"target_type", "revenue_type", "split", "estimator", "predictor", "actual_bracket",
                    "predicted_bracket", "model", "status", "evaluation_split", "bracket"}
for section, variable in summary_tables.items():
    frame = globals().get(variable, pd.DataFrame())
    if not isinstance(frame, pd.DataFrame) or frame.empty:
        continue
    for record in frame.to_dict("records"):
        identity = {k: v for k, v in record.items() if k in identity_columns and pd.notna(v)}
        target_type = identity.get("target_type", identity.get("revenue_type"))
        summary_split = identity.get("split", identity.get("evaluation_split"))
        if summary_split is None and section in {"bracket_summary", "bracket_recall"}:
            summary_split = "test"
        estimator = identity.get("estimator", identity.get("predictor", identity.get("model")))
        detail = json.dumps({k: str(v) for k, v in record.items()
                             if not isinstance(v, (int, float, np.number))}, default=str)
        any_metric = False
        for key, value in record.items():
            if key in identity_columns or isinstance(value, (bool, np.bool_)) or not isinstance(value, (int, float, np.number)):
                continue
            if not np.isfinite(value):
                continue
            summary_rows.append({"section": section, "target_type": target_type,
                                 "split": summary_split, "estimator": estimator,
                                 "bracket": identity.get("actual_bracket", identity.get("bracket")), "metric": key,
                                 "value": float(value), "detail": detail})
            any_metric = True
        if not any_metric:
            summary_rows.append({"section": section, "target_type": target_type,
                                 "split": summary_split, "estimator": estimator,
                                 "bracket": identity.get("actual_bracket", identity.get("bracket")), "metric": "status",
                                 "value": None, "detail": json.dumps(record, default=str)})
for row in debug_findings_pdf.to_dict("records"):
    summary_rows.append({"section": "finding", "target_type": None, "split": None, "estimator": None,
                         "bracket": None, "metric": row["area"], "value": None,
                         "detail": f"[{row['severity']}] {row['evidence']} Next: {row['next_check']}"})
for row in debug_run_provenance_pdf.to_dict("records"):
    summary_rows.append({"section": "provenance", "target_type": None, "split": None, "estimator": None,
                         "bracket": None, "metric": "selected_run", "value": None,
                         "detail": json.dumps(row, default=str)})
debug_summary_pdf = pd.DataFrame(summary_rows)
debug_summary_pdf["mlflow_parent_run_id"] = debug_run_id
debug_summary_pdf["horizon_months"] = DEBUG_HORIZON
debug_summary_pdf["historical_test_cutoff"] = DEBUG_TEST_CUTOFF
if not debug_predictions_pdf.empty and "experiment_id" in debug_predictions_pdf:
    ids = debug_predictions_pdf["experiment_id"].dropna().unique()
    debug_summary_pdf["experiment_id"] = str(ids[0]) if len(ids) == 1 else None

print("Findings below are diagnostic flags. Compressed predictions alone do not prove underfitting.")
print("Download debug_summary_pdf as CSV and share it for the next review.")
print("The summary includes train/test errors, spread, rank correlation, bracket recall, tuning status and source coverage.")
display(debug_findings_pdf)
display(debug_summary_pdf)

debug_report_tables = {
    "summary": debug_summary_pdf,
    "findings": debug_findings_pdf,
    "run_provenance": debug_run_provenance_pdf,
    "source_load_status": debug_source_load_pdf,
    "launch_cohorts": debug_cohort_summary_pdf,
}
for label, variable in {
    "regression_metrics": "debug_regression_metrics_pdf",
    "prediction_spread": "debug_prediction_spread_pdf",
    "bracket_accuracy": "debug_bracket_summary_pdf",
    "bracket_recall_precision": "debug_bracket_recall_pdf",
    "bracket_confusion": "debug_bracket_confusion_pdf",
    "worst_test_games": "debug_worst_games_pdf",
    "per_game_diagnostics": "debug_prediction_diagnostics_pdf",
    "target_coverage": "debug_target_coverage_pdf",
    "rebuilt_historical_targets": "debug_historical_targets_pdf",
    "target_differences": "debug_target_mismatches_pdf",
    "gap_examples": "debug_gap_examples_pdf",
    "rebuilt_monthly_actuals": "debug_rebuilt_monthly_pdf",
    "source_conflicts": "debug_source_conflicts_pdf",
    "source_input_issues": "debug_source_input_issues_pdf",
    "split_checks": "debug_split_audit_pdf",
    "feature_quality": "debug_feature_quality_pdf",
    "feature_shift": "debug_feature_shift_pdf",
    "feature_associations_train_only": "debug_feature_associations_pdf",
    "duplicate_feature_vectors": "debug_duplicate_feature_vectors_pdf",
    "tuning_audit": "debug_tuning_audit_pdf",
}.items():
    frame = globals().get(variable)
    if isinstance(frame, pd.DataFrame):
        debug_report_tables[label] = frame
try:
    names = sorted(debug_report_tables)
    selected_table = "summary"
    try:
        previous = dbutils.widgets.get("debug_historical_table")
        if previous in names:
            selected_table = previous
    except Exception:
        pass
    dbutils.widgets.dropdown("debug_historical_table", selected_table, names, "Debug table to view / download")
    selected_table = dbutils.widgets.get("debug_historical_table")
    if selected_table in debug_report_tables:
        print(f"Selected table: {selected_table}. Change the selector and rerun this cell to view another table.")
        display(debug_report_tables[selected_table])
except NameError:
    pass  # Running outside Databricks: the tables are still available as pandas DataFrames.
